# VS Code Extension Test Debugging Guide

This notebook provides a comprehensive guide for debugging VS Code extension test failures, specifically focusing on:

- **API Proposal Configuration Issues**: Missing or incorrect `enabledApiProposals` in package.json
- **Test Framework Conflicts**: Vitest/Mocha integration problems 
- **Extension Host Process Failures**: Common errors when running VS Code extension tests
- **Module Loading Problems**: Import statement and dependency resolution issues

## Current Issue Analysis

The VS Code extension test for `HostServiceImpl` is failing due to multiple configuration issues:

1. **API Proposals Not Loading**: Despite having `chatParticipantPrivate` in package.json, VS Code claims it's missing
2. **Vitest/Mocha Conflict**: Test bundle includes both Vitest and Mocha test frameworks causing conflicts
3. **Extension Development Mode**: Extension requires development mode or `--enable-proposed-api` flag

## 1. Setting Up the Environment

Import necessary libraries and set up the debugging environment for analyzing VS Code extension test failures.

In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Optional

# Set up working directory
WORKSPACE_ROOT = Path("/Users/guo/OSS/vscode-copilot-chat")
PACKAGE_JSON_PATH = WORKSPACE_ROOT / "package.json"

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Package.json exists: {PACKAGE_JSON_PATH.exists()}")

# Helper functions for debugging VS Code extension issues
def load_package_json() -> Dict[str, Any]:
    """Load and parse package.json"""
    with open(PACKAGE_JSON_PATH, 'r') as f:
        return json.load(f)

def analyze_api_proposals(pkg: Dict[str, Any]) -> Dict[str, Any]:
    """Analyze enabledApiProposals configuration"""
    proposals = pkg.get('enabledApiProposals', [])
    return {
        'total_count': len(proposals),
        'proposals': proposals,
        'has_chat_participant_private': 'chatParticipantPrivate' in proposals,
        'has_contrib_language_model_tool_sets': 'contribLanguageModelToolSets' in proposals,
        'versioned_proposals': [p for p in proposals if '@' in p]
    }

def find_test_files(pattern: str = "*.test.ts") -> List[Path]:
    """Find test files matching pattern"""
    return list(WORKSPACE_ROOT.rglob(pattern))

print("Environment setup complete!")

## 2. Analyzing Test Output and Error Messages

Parse and analyze error messages from VS Code extension test output to identify the root causes of failures.

In [ ]:
# Current test error output analysis
test_error_output = """
Extension 'goastro.copilot-x CANNOT USE these API proposals 'extensionsAny, newSymbolNamesProvider, interactive, codeActionAI, activeComment, commentReveal, contribCommentThreadAdditionalMenu, contribCommentsViewThreadMenus, documentFiltersExclusive, embeddings, findTextInFiles, findTextInFiles2, findFiles2, textSearchProvider, terminalDataWriteEvent, terminalExecuteCommandEvent, terminalSelection, terminalQuickFixProvider, mappedEditsProvider, aiRelatedInformation, chatParticipantAdditions, chatEditing, defaultChatParticipant, contribSourceControlInputBoxMenu, authLearnMore, testObserver, aiTextSearchProvider, chatParticipantPrivate, chatProvider, contribDebugCreateConfiguration, chatReferenceDiagnostic, textSearchProvider2, chatReferenceBinaryData, languageModelSystem, languageModelCapabilities, inlineCompletionsAdditions, languageModelDataPart, chatStatusItem, taskProblemMatcherStatus, contribLanguageModelToolSets, textDocumentChangeReason, resolvers'. You MUST start in extension development mode or use the --enable-proposed-api command line flag

Extension 'goastro.copilot-x' CANNOT use API proposal: chatParticipantPrivate.
Its package.json#enabledApiProposals-property declares:  but NOT chatParticipantPrivate.

Error: Vitest failed to access its internal state.
"""

def parse_test_errors(output: str) -> Dict[str, List[str]]:
    """Parse test error output to identify different types of issues"""
    issues = {
        'api_proposal_errors': [],
        'vitest_errors': [],
        'extension_loading_errors': [],
        'missing_proposals': []
    }

    lines = output.strip().split('\n')
    for line in lines:
        if 'CANNOT USE these API proposals' in line:
            issues['api_proposal_errors'].append(line.strip())
        elif 'Vitest failed to access' in line:
            issues['vitest_errors'].append(line.strip())
        elif 'Extension' in line and 'CANNOT' in line:
            issues['extension_loading_errors'].append(line.strip())
        elif 'enabledApiProposals-property declares:  but NOT' in line:
            issues['missing_proposals'].append(line.strip())

    return issues

# Analyze the current error output
error_analysis = parse_test_errors(test_error_output)

print("=== Error Analysis Results ===")
for category, errors in error_analysis.items():
    print(f"\n{category.upper()}:")
    for error in errors:
        print(f"  - {error}")

# Key insights
print("\n=== Key Insights ===")
print("1. Extension needs development mode or --enable-proposed-api flag")
print("2. VS Code claims enabledApiProposals is empty but package.json has content")
print("3. Vitest is being imported in test bundle causing conflicts with Mocha")
print("4. Extension cannot register language model tools due to missing API proposal")

## 3. Identifying Common Extension API Issues

Detect and categorize common API proposal errors, missing enabledApiProposals, and extension development mode requirements.

In [ ]:
# Load and analyze current package.json
pkg = load_package_json()
api_analysis = analyze_api_proposals(pkg)

print("=== Current API Proposals Analysis ===")
print(f"Total API proposals: {api_analysis['total_count']}")
print(f"Has chatParticipantPrivate: {api_analysis['has_chat_participant_private']}")
print(f"Has contribLanguageModelToolSets: {api_analysis['has_contrib_language_model_tool_sets']}")
print(f"Versioned proposals: {api_analysis['versioned_proposals']}")

print("\n=== All API Proposals ===")
for i, proposal in enumerate(api_analysis['proposals'], 1):
    print(f"{i:2d}. {proposal}")

# Check for common issues
print("\n=== Issue Detection ===")
issues_found = []

if api_analysis['versioned_proposals']:
    issues_found.append(f"Found {len(api_analysis['versioned_proposals'])} versioned API proposals that may not exist")

if not api_analysis['has_chat_participant_private']:
    issues_found.append("Missing 'chatParticipantPrivate' API proposal")

if not api_analysis['has_contrib_language_model_tool_sets']:
    issues_found.append("Missing 'contribLanguageModelToolSets' API proposal")

if issues_found:
    print("Issues found:")
    for issue in issues_found:
        print(f"  ❌ {issue}")
else:
    print("  ✅ No obvious API proposal issues detected")

# The discrepancy: VS Code says empty but package.json has content
print("\n=== Key Problem ===")
print("🔍 VS Code reports 'enabledApiProposals-property declares:  but NOT chatParticipantPrivate'")
print("📋 But package.json clearly contains chatParticipantPrivate in the proposals list")
print("💡 This suggests the compiled extension bundle may not include the correct package.json")

## 4. Debugging Test Configuration Problems

Analyze package.json configuration issues, test runner setup problems, and extension manifest validation errors.

In [ ]:
# Check test configuration files
test_config_files = [
    WORKSPACE_ROOT / ".vscode-test.mjs",
    WORKSPACE_ROOT / "vite.config.ts",
    WORKSPACE_ROOT / "tsconfig.json"
]

print("=== Test Configuration Files ===")
for config_file in test_config_files:
    exists = config_file.exists()
    print(f"{'✅' if exists else '❌'} {config_file.name}: {'Found' if exists else 'Missing'}")

# Find test files and categorize them
test_files = find_test_files()
vitest_files = []
mocha_files = []
other_files = []

print(f"\n=== Test Files Analysis ===")
print(f"Total test files found: {len(test_files)}")

for test_file in test_files:
    try:
        content = test_file.read_text(encoding='utf-8')
        if 'from vitest' in content or 'import { describe, it, expect' in content:
            vitest_files.append(test_file)
        elif 'suite(' in content or 'test(' in content:
            mocha_files.append(test_file)
        else:
            other_files.append(test_file)
    except Exception as e:
        print(f"Error reading {test_file}: {e}")

print(f"Vitest-based test files: {len(vitest_files)}")
print(f"Mocha-based test files: {len(mocha_files)}")
print(f"Other test files: {len(other_files)}")

if vitest_files:
    print("\n=== Vitest Files (Causing Conflicts) ===")
    for vf in vitest_files[:5]:  # Show first 5
        rel_path = vf.relative_to(WORKSPACE_ROOT)
        print(f"  📄 {rel_path}")
    if len(vitest_files) > 5:
        print(f"  ... and {len(vitest_files) - 5} more")

# Check for the specific test file mentioned in the error
workbench_test = WORKSPACE_ROOT / "src/platform/workbench/test/vscode-node/workbenchServiceImpl.test.ts"
print(f"\n=== Target Test File ===")
print(f"HostServiceImpl test exists: {workbench_test.exists()}")

if workbench_test.exists():
    content = workbench_test.read_text()
    uses_mocha = 'suite(' in content and 'test(' in content
    uses_assert = 'import * as assert' in content
    print(f"Uses Mocha syntax: {uses_mocha}")
    print(f"Uses Node.js assert: {uses_assert}")
    print(f"✅ This test file looks correctly configured for VS Code extension testing")

## 5. Fixing Package.json and API Proposal Issues

Generate solutions for missing API proposals in package.json and provide recommendations for fixing extension configuration.

In [ ]:
# Generate solutions for API proposal issues
def generate_api_proposal_fixes(pkg: Dict[str, Any]) -> Dict[str, Any]:
    """Generate fixes for API proposal configuration"""
    proposals = pkg.get('enabledApiProposals', [])
    fixes = {
        'remove_versions': False,
        'missing_proposals': [],
        'cleaned_proposals': []
    }

    # Clean versioned proposals
    cleaned = []
    for proposal in proposals:
        if '@' in proposal:
            fixes['remove_versions'] = True
            clean_proposal = proposal.split('@')[0]
            cleaned.append(clean_proposal)
        else:
            cleaned.append(proposal)

    fixes['cleaned_proposals'] = cleaned

    # Check for essential proposals
    essential_proposals = [
        'chatParticipantPrivate',
        'contribLanguageModelToolSets',
        'languageModelSystem',
        'chatProvider'
    ]

    for essential in essential_proposals:
        if essential not in cleaned:
            fixes['missing_proposals'].append(essential)

    return fixes

# Analyze current setup
fixes = generate_api_proposal_fixes(pkg)

print("=== API Proposal Fix Analysis ===")
print(f"Need to remove version suffixes: {fixes['remove_versions']}")
print(f"Missing essential proposals: {len(fixes['missing_proposals'])}")

if fixes['missing_proposals']:
    print("\nMissing proposals:")
    for missing in fixes['missing_proposals']:
        print(f"  ❌ {missing}")

print(f"\nCleaned proposals count: {len(fixes['cleaned_proposals'])}")

# Show the fix script
print("\n=== Recommended Fix Script ===")
fix_script = f'''
import json

# Load package.json
with open('package.json', 'r') as f:
    data = json.load(f)

# Clean API proposals (remove version suffixes)
proposals = data['enabledApiProposals']
cleaned_proposals = []
for proposal in proposals:
    clean_proposal = proposal.split('@')[0]
    cleaned_proposals.append(clean_proposal)

# Update package.json
data['enabledApiProposals'] = cleaned_proposals

# Save back to file
with open('package.json', 'w') as f:
    json.dump(data, f, indent='\\t')

print("Fixed API proposals in package.json")
'''

print(fix_script)

print("\n=== Alternative Solution: Test with Development Mode ===")
print("Run tests with the --enable-proposed-api flag:")
print("vscode-test --enable-proposed-api goastro.copilot-x --grep 'HostServiceImpl'")

print("\n=== Root Cause Analysis ===")
print("The issue appears to be that VS Code is not recognizing the API proposals")
print("even though they exist in package.json. This could be due to:")
print("1. Extension not running in development mode")
print("2. Build process not preserving package.json correctly")
print("3. Test runner configuration issues")
print("4. VS Code version compatibility with proposed APIs")

## 6. Working with VS Code Test Framework

Debug issues with Mocha test runner, Vitest integration problems, and extension host process failures.

In [ ]:
# Analyze Vitest/Mocha conflict and provide solutions
def analyze_test_framework_conflict():
    """Analyze the Vitest/Mocha conflict issue"""

    print("=== VS Code Test Framework Analysis ===")
    print("VS Code extension tests must use Mocha, not Vitest")
    print("The error 'Vitest failed to access its internal state' indicates:")
    print("  1. Vitest is being imported in the test bundle")
    print("  2. This conflicts with VS Code's expected Mocha test runner")
    print("  3. The test bundle includes both frameworks")

    print("\n=== Conflict Sources ===")
    if vitest_files:
        print(f"Found {len(vitest_files)} files importing from Vitest:")
        for vf in vitest_files:
            rel_path = vf.relative_to(WORKSPACE_ROOT)
            print(f"  📄 {rel_path}")

    print("\n=== Solutions ===")
    print("1. TEMPORARY: Exclude Vitest test files from bundle")
    print("   - Rename .test.ts files to .test.ts.vitest.bak")
    print("   - Rebuild extension bundle")
    print("   - Run tests")

    print("\n2. PERMANENT: Convert Vitest tests to Mocha")
    print("   - Replace: import { describe, it, expect, vi } from 'vitest'")
    print("   - With: import * as assert from 'assert'")
    print("   - Replace: describe() → suite()")
    print("   - Replace: it() → test()")
    print("   - Replace: expect().toBe() → assert.strictEqual()")
    print("   - Replace: vi.fn() → sinon.stub() or manual mocks")

    print("\n3. BUNDLE CONFIGURATION: Update build to exclude Vitest")
    print("   - Modify .esbuild.ts to exclude Vitest test files")
    print("   - Add patterns to ignore Vitest imports")

# Run the analysis
analyze_test_framework_conflict()

# Generate commands to temporarily fix the issue
print("\n=== Quick Fix Commands ===")
print("# Find and temporarily rename Vitest test files:")
vitest_commands = []
for vf in vitest_files:
    rel_path = vf.relative_to(WORKSPACE_ROOT)
    cmd = f'mv "{rel_path}" "{rel_path}.vitest.bak"'
    vitest_commands.append(cmd)

if vitest_commands:
    print("cd /Users/guo/OSS/vscode-copilot-chat")
    for cmd in vitest_commands[:3]:  # Show first 3
        print(cmd)
    if len(vitest_commands) > 3:
        print(f"# ... and {len(vitest_commands) - 3} more files")

print("\n# Rebuild extension:")
print("npm run compile")

print("\n# Run specific test:")
print("vscode-test --grep 'HostServiceImpl'")

print("\n# Restore files after testing:")
print("# git checkout -- src/")

print("\n=== Test Configuration Best Practices ===")
print("✅ Use Mocha syntax for VS Code extension tests")
print("✅ Import assert from Node.js standard library")
print("✅ Use suite() and test() functions")
print("✅ Avoid mixing test frameworks in the same project")
print("✅ Separate unit tests (Vitest) from integration tests (Mocha)")

## 7. Resolving Module Loading Errors

Identify and fix module loading issues, import statement problems, and dependency resolution errors in extension tests.

In [ ]:
# Comprehensive module loading diagnostics
def diagnose_module_loading_issues():
    """Diagnose and provide fixes for module loading issues"""

    print("=== Module Loading Diagnostics ===")

    # Check the specific test file import
    workbench_test = WORKSPACE_ROOT / "src/platform/workbench/test/vscode-node/workbenchServiceImpl.test.ts"
    if workbench_test.exists():
        content = workbench_test.read_text()
        print("Target test file analysis:")

        # Check import statements
        import_lines = [line.strip() for line in content.split('\n') if line.strip().startswith('import')]
        for imp in import_lines:
            print(f"  📦 {imp}")

        # Check if the imported module exists
        import_path = "../../vscode/workbenchServiceImpt"
        implied_file = WORKSPACE_ROOT / "src/platform/workbench/vscode/workbenchServiceImpt.ts"
        print(f"\nImported module check:")
        print(f"  Import path: {import_path}")
        print(f"  Resolved file: {implied_file}")
        print(f"  File exists: {implied_file.exists()}")

        if implied_file.exists():
            print("  ✅ Import looks correct")
        else:
            print("  ❌ Import path may be incorrect")

    print("\n=== Common Module Loading Issues ===")
    common_issues = [
        {
            'issue': 'Incorrect relative import paths',
            'solution': 'Verify ../../../ style paths match actual directory structure'
        },
        {
            'issue': 'Missing file extensions in imports',
            'solution': 'VS Code extension tests may require explicit .js extensions in some cases'
        },
        {
            'issue': 'TypeScript compilation issues',
            'solution': 'Ensure tsconfig.json includes all test files and paths are correct'
        },
        {
            'issue': 'Bundle configuration excluding test files',
            'solution': 'Check .esbuild.ts configuration for test file inclusion'
        }
    ]

    for issue_info in common_issues:
        print(f"  🔍 {issue_info['issue']}")
        print(f"     💡 {issue_info['solution']}")

# Run diagnostics
diagnose_module_loading_issues()

print("\n" + "="*60)
print("                 SUMMARY & RECOMMENDATIONS")
print("="*60)

print("\n🎯 PRIMARY ISSUES IDENTIFIED:")
print("1. ❌ API Proposals Configuration")
print("   - VS Code reports empty enabledApiProposals despite package.json content")
print("   - Extension requires development mode or --enable-proposed-api flag")

print("\n2. ❌ Test Framework Conflict")
print("   - Vitest imports in test bundle conflict with Mocha test runner")
print("   - VS Code extension tests must use Mocha exclusively")

print("\n3. ❌ Extension Development Mode")
print("   - Extension needs development mode to access proposed APIs")
print("   - Test runner may need specific flags")

print("\n🔧 IMMEDIATE ACTIONS:")
print("1. Temporarily exclude Vitest test files from bundle")
print("2. Ensure package.json API proposals are clean (no @version suffixes)")
print("3. Run tests with --enable-proposed-api flag")
print("4. Verify HostServiceImpl test uses correct Mocha syntax")

print("\n🏁 TEST COMMAND TO TRY:")
print("cd /Users/guo/OSS/vscode-copilot-chat")
print("# Rename Vitest files temporarily")
print("find src -name '*.test.ts' -exec grep -l 'from.*vitest' {} \\; | xargs -I {} mv {} {}.bak")
print("# Recompile")
print("npm run compile")
print("# Run test with API proposals enabled")
print("vscode-test --enable-proposed-api goastro.copilot-x --grep 'HostServiceImpl'")

print("\n📋 LONG-TERM FIXES:")
print("- Convert Vitest tests to Mocha for VS Code compatibility")
print("- Separate unit tests (Vitest) from integration tests (Mocha)")
print("- Update build configuration to handle test framework separation")
print("- Ensure all API proposals are properly configured for production")

print(f"\n✅ Notebook analysis complete!")
print(f"📊 Found {len(test_files)} test files, {len(vitest_files)} using Vitest")
print(f"🎯 Target test file exists and uses correct Mocha syntax")
print(f"📦 Package.json has {api_analysis['total_count']} API proposals configured")